# Step 6 — manual stop additions

Stations added by hand on top of step 5's night train stops, so the catalog
covers places no night train serves *today* but a target network would: large
urban areas without a qualified stop, tourism regions, and major ferry hubs.

**This notebook is the source of truth for those additions.** The selections
live in the `ADDITIONS` cells below, grouped by region and keyed by OSM stop
id, and everything else (name, coordinates, country) is looked up from step 3b
and step 4 at run time — so a stop is added or removed by editing one line
here, and `git diff` shows exactly what changed and why.

## The `reason` field

Every addition carries a `reason` of the form `criterion` or `criterion:place`.
The design requires that *"why is station X (not) included?"* be answerable
from the data alone, including by people outside the project. The vocabulary:

| reason | when |
|---|---|
| `fua:<city>` | functional urban area with no qualified stop — name the city |
| `tourism:<region>` | tourism destination — name the region |
| `ferry:<port>` | major ferry hub — name the port |
| `border` | border/interchange station |
| `network` | needed to make a corridor coherent |
| `night_train_stop` | served today but missing from ONTD/step 5 |

`<city>` is a placeholder to **replace**, not to keep: `fua:Bayreuth`, not
`fua:<city>`. The report at the bottom counts placeholders as unfilled.

## The `infra_versions` field

Every addition also says **which infrastructure version(s) the stop belongs
to**, as a third tuple element. Two-element entries mean both — that is the
default and covers every existing station. Use the constants:

| value | meaning |
|---|---|
| `INFRA_BOTH` (`infra-2026;infra-2032`) | exists today and stays — the default; omit the third element |
| `INFRA_2032` (`infra-2032`) | not built yet — Rail Baltica stations, a future HSL station |
| `INFRA_2026` (`infra-2026`) | exists today but is not part of the 2032 network (a station a line upgrade bypasses) |

Step 5 stops are `INFRA_BOTH` by definition. The value is tagged here and
carried through step 10 into the catalog; **seed.py does not act on it yet**
(every stop still seeds into every snapshot version) — the tag is the
calibration record, the seed-side consumption is a later work package.

`infra-2026` / `infra-2032` are the two routing graphs (`routing_graph_key`
in `scenario.scenarios`, `COMPOSE_PROFILES=infra-2032`). They are **not**
`stop_infra_version`, which is the integer snapshot number of the stop table.

## Finding OSM ids for a batch — step 6a

`step6a_resolve_candidates.py` takes a CSV of names and coordinates and
prints paste-ready lines for these dicts, matching against step 3b the way
step 5 matches the schedule and applying the guards below before printing.
`step6_gap_closure_2026-09.csv` is the record of the September batch and the
template for the next one.

## Guards

The resolve cell refuses to write the output while any addition

- points at an OSM id step 3b does not know,
- points at a metro/tram/bus/funicular object instead of the railway station
  (`station_mode` check; ferry piers pass only with a `ferry:` reason), or
- duplicates a step 5 stop — same OSM id, same ONTD station through a
  different OSM object, or within 300 m of a qualified stop.

Those three are how the catalog once ended up with two Gesundbrunnens; they
are errors, not judgement calls. What *is* a judgement call — an `fua:` reason
where the same urban area meanwhile has a qualified stop — is written to
`data/step6_overlap_review.csv` for review instead of blocking the run.

## Output

`data/step6_manual_additions.csv` — `stop_id, stop_name, country, stop_lat,
stop_lon, reason, infra_versions` — consumed by `step10_export_seed_stops.py`, which unions it
with the current step 5 output.


In [ ]:
import csv
import math
from collections import Counter

from data_sources import DATA_DIR, ensure_local, local_input

OUTPUT_PATH = DATA_DIR / "step6_manual_additions.csv"
OVERLAP_REVIEW_PATH = DATA_DIR / "step6_overlap_review.csv"
# Infrastructure versions a stop can belong to (see the header). Two-element
# entries default to INFRA_BOTH.
INFRA_2026 = "infra-2026"
INFRA_2032 = "infra-2032"
INFRA_BOTH = f"{INFRA_2026};{INFRA_2032}"
INFRA_VALID = {INFRA_2026, INFRA_2032}

## The additions

One dict per region, `stop_id: (name, reason)`. The name is a comment for
readability only — it is re-read from step 3b when the file is written, so a
stale name here cannot corrupt the output. Add a stop by adding a line; remove
one by deleting its line.

An entry is `stop_id: (name, reason)` or `stop_id: (name, reason,
infra_versions)`; leave the third element out unless the stop is
`INFRA_2032`-only or `INFRA_2026`-only.


In [2]:
# Germany, Austria, Switzerland
ADDITIONS_GERMANY = {
    # --- DE ---
    "osm:n31485922": ("Bayreuth Hbf", "fua:<city>"),
    "osm:n1874501382": ("Bielefeld Hauptbahnhof", "fua:<city>"),
    "osm:w24806780": ("Braunschweig Hauptbahnhof", "fua:<city>"),
    "osm:n26562398": ("Bremerhaven Hauptbahnhof", "fua:<city>"),
    "osm:n2711388096": ("Böblingen", "fua:<city>"),
    "osm:n3607858763": ("Chemnitz Hauptbahnhof", "fua:<city>"),
    "osm:n2599505466": (
        "Cottbus Hauptbahnhof / Chóśebuz głowne dwórnišćo",
        "fua:<city>",
    ),
    "osm:n4189000814": ("Flensburg / Flensborg", "fua:<city>"),
    "osm:n1840958277": ("Gera Hauptbahnhof", "fua:<city>"),
    "osm:n1438696887": ("Görlitz", "fua:<city>"),
    "osm:n3450444902": ("Gütersloh Hbf", "fua:<city>"),
    "osm:n27385328": ("Heilbronn Hauptbahnhof", "fua:<city>"),
    "osm:n3616040153": ("Hildesheim Hauptbahnhof", "fua:<city>"),
    "osm:n1126168394": ("Jena Paradies", "fua:<city>"),
    "osm:n30959690": ("Kaiserslautern Hauptbahnhof", "fua:<city>"),
    "osm:n4530820004": ("Kempten (Allgäu) Hbf", "fua:<city>"),
    "osm:n4257641280": ("Kiel Hauptbahnhof", "fua:<city>"),
    "osm:n534753716": ("Landshut (Bay) Hbf", "fua:<city>"),
    "osm:n3087634633": ("Magdeburg Hauptbahnhof", "fua:<city>"),
    "osm:n7160009313": ("Neumünster", "fua:<city>"),
    "osm:n91753264": ("Oldenburg (Oldb) Hbf", "fua:<city>"),
    "osm:n268894281": ("Osnabrück Hauptbahnhof", "fua:<city>"),
    "osm:n2675283037": ("Paderborn Hauptbahnhof", "fua:<city>"),
    "osm:n25972727": ("Pforzheim Hauptbahnhof", "fua:<city>"),
    "osm:n1755712810": ("Plauen (Vogtl) ob Bf", "fua:<city>"),
    "osm:n25233549": ("Reutlingen Hbf", "fua:<city>"),
    "osm:n987773654": ("Rostock Hauptbahnhof", "fua:<city>"),
    "osm:n259449966": ("Saarbrücken Hauptbahnhof", "fua:<city>"),
    "osm:n27381920": ("Schweinfurt Hbf", "fua:<city>"),
    "osm:n252098248": ("Schwerin Hauptbahnhof", "fua:<city>"),
    "osm:n277350630": ("Stralsund Hbf", "tourism:<region>"),
    "osm:n745097775": ("Trier Hbf", "fua:<city>"),
    "osm:n338899629": ("Wolfsburg Hauptbahnhof", "fua:<city>"),
    # --- CH ---
    "osm:n2051794005": ("Luzern", "fua:<city>"),
    "osm:n2428167137": ("Schaffhausen", "fua:<city>"),
    "osm:n1346888802": ("St. Gallen", "fua:<city>"),
    "osm:n3081154442": ("Thun", "fua:<city>"),
    "osm:n1286602751": ("Winterthur", "fua:<city>"),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- AT --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n3508515531": ("St. Michael", "network"),  # Südbahn/Schoberpass junction
    "osm:n3509946492": ("Selzthal", "network"),  # Ennstal/Pyhrn junction
    "osm:n3244697567": (
        "Baden",
        "tourism:Baden bei Wien",
    ),  # the schedule name 'Baden' collapses two stations (README, still open); this is the AT one
    # --- CH --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n3080695733": ("Baden", "network"),  # the CH one; Zürich–Basel line
    "osm:n3080746008": ("Aarau", "fua:Aarau"),
    "osm:n636798044": ("Olten", "network"),  # CH's central junction
    "osm:n3080695737": ("Brugg", "network"),  # Zürich–Basel/Bözberg junction
    "osm:n1280652702": ("Zug", "fua:Zug"),
    "osm:n1800313662": ("Lausanne", "fua:Lausanne"),
    "osm:n4890696636": ("Genève", "fua:Genève"),  # Genève-Cornavin
    "osm:n3081154375": ("Fribourg/Freiburg", "fua:Fribourg"),
    # --- DE --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n205364328": (
        "Frankfurt (Main) Hauptbahnhof",
        "fua:Frankfurt am Main",
    ),  # hidden by the 2 km check: Süd is 1.8 km away. THE main station.; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n459277140": (
        "Berlin Gesundbrunnen",
        "network",
    ),  # removed 2026-08-24 as duplicate of step 5; step 5 lost it 2026-08-28; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n3419894993": (
        "Berlin Ostbahnhof",
        "network",
    ),  # Berlin already has Hbf/Südkreuz qualified; second-tier Berlin stations are corridor stations; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n29805967": (
        "Berlin-Lichtenberg",
        "network",
    ),  # eastbound terminus (Warszawa/Kyiv trains); id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n3101920259": (
        "Berlin-Spandau",
        "network",
    ),  # westbound stop of Berlin–NRW/Paris trains; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n74880878": (
        "Köln Messe/Deutz",
        "network",
    ),  # HSL-side station; Köln Hbf 1.1 km — distinct station, tracks not shared; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n1458488633": (
        "Dortmund Hbf",
        "fua:Dortmund",
    ),  # id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n25149874": (
        "Lübeck Hauptbahnhof",
        "fua:Lübeck",
    ),  # id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n2703868858": (
        "Regensburg Hauptbahnhof",
        "fua:Regensburg",
    ),  # id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n2835627177": (
        "Bamberg",
        "fua:Bamberg",
    ),  # id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n2821098770": (
        "Gießen",
        "fua:Gießen",
    ),  # id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n269763316": (
        "Siegen Hauptbahnhof",
        "fua:Siegen",
    ),  # id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n2631683231": (
        "Mönchengladbach Hbf",
        "fua:Mönchengladbach",
    ),  # id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n35879657": (
        "Siegburg/Bonn",
        "network",
    ),  # ICE station on the Köln–Rhein/Main HSL; Bonn Hbf is on the classic line. Export coordinate 2.7 km off; corrected. id from charges/sources/de_station_charges.csv
    "osm:n14334692": (
        "Passau Hbf",
        "border",
    ),  # DE/AT; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n321555238": (
        "Frankfurt (Oder)",
        "border",
    ),  # DE/PL; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n3620887012": (
        "Wittenberge",
        "network",
    ),  # Berlin–Hamburg junction; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n620379882": (
        "Bad Oldesloe",
        "network",
    ),  # Hamburg–Lübeck line; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n6771321243": (
        "Neustadt (Holstein)",
        "network",
    ),  # Vogelfluglinie; Fehmarnbelt route from ~2029; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n253313309": (
        "Oldenburg (Holstein)",
        "network",
    ),  # Vogelfluglinie; Fehmarnbelt route from ~2029; id from charges/sources/de_station_charges.csv (old catalog id)
    "osm:n6069304543": (
        "Timmendorfer Strand",
        "tourism:Ostsee",
    ),  # export coordinate was St. Pölten — wrong; corrected
    "osm:n21322676": (
        "Rastatt",
        "network",
    ),  # export coordinate was Wuppertal — wrong; corrected. Rheintalbahn
    "osm:n4565521156": (
        "Remscheid-Lennep",
        "network",
    ),  # removed 2026-08-24 as duplicate of step 5; step 5 lost it 2026-08-28. The frozen export's 'Rastatt' row actually carried THIS station's coordinate. id from charges/sources/de_station_charges.csv
}

In [3]:
# France, Benelux
ADDITIONS_FRANCE = {
    "osm:n269296749": ("Marne-la-Vallée Chessy", "tourism:Disneyland Paris"),
    # --- FR ---
    "osm:n4290854846": ("Aix-en-Provence", "fua:<city>"),
    "osm:n1680885216": ("Amiens", "fua:<city>"),
    "osm:n3486353293": ("Angers Saint-Laud", "fua:<city>"),
    "osm:n5061961433": ("Annecy", "fua:<city>"),
    "osm:n7167504997": ("Arras", "fua:<city>"),
    "osm:n3805976209": ("Avignon-Centre", "fua:<city>"),
    "osm:n2501252269": ("Belfort", "fua:<city>"),
    "osm:n2500070617": ("Besançon-Viotte", "fua:<city>"),
    "osm:n8213648712": ("Boulogne Ville", "fua:<city>"),
    "osm:n194212267": ("Bourges", "fua:<city>"),
    "osm:n2207570062": ("Brest", "tourism:<region>"),
    "osm:n5598384401": ("Caen", "fua:<city>"),
    "osm:n9183304884": ("Calais-Ville", "fua:<city>"),
    "osm:n5066478129": ("Chambéry - Challes-les-Eaux", "fua:<city>"),
    "osm:n8303862255": ("Chartres", "fua:<city>"),
    "osm:w112095036": ("Cherbourg", "fua:<city>"),
    "osm:n10936459654": ("Clermont-Ferrand", "fua:<city>"),
    "osm:n398836628": ("Colmar", "fua:<city>"),
    "osm:n11556100824": ("Douai", "fua:<city>"),
    "osm:n312805118": ("Dunkerque", "fua:<city>"),
    "osm:n5070332503": ("Grenoble", "fua:<city>"),
    "osm:n4975049664": ("La Rochelle", "fua:<city>"),
    "osm:n847701543": ("Le Havre", "fua:<city>"),
    "osm:n8745537419": ("Lille-Flandres", "fua:<city>"),
    "osm:n3471044194": ("Limoges-Bénédictins", "fua:<city>"),
    "osm:n10940619366": ("Lorient", "fua:<city>"),
    "osm:n4290857018": ("Lyon Perrache", "fua:<city>"),
    "osm:n11225690169": ("Martigues", "fua:<city>"),
    "osm:n4290857026": ("Metz", "fua:<city>"),
    "osm:n4225150278": ("Montbéliard", "fua:<city>"),
    "osm:n2502268309": ("Mulhouse-Ville", "fua:<city>"),
    "osm:n4290857032": ("Nancy", "fua:<city>"),
    "osm:n3486337796": ("Nantes", "fua:<city>"),
    "osm:n9912502487": ("Poitiers", "fua:<city>"),
    "osm:n4986753876": ("Quimper", "fua:<city>"),
    "osm:n2517400258": ("Reims", "fua:<city>"),
    "osm:n4250849558": ("Rennes", "fua:<city>"),
    "osm:n2483753165": ("Roanne", "fua:<city>"),
    "osm:n2076751841": ("Rouen Rive-Droite", "fua:<city>"),
    "osm:n4280767167": ("Saint-Brieuc", "fua:<city>"),
    "osm:n829527258": ("Saint-Raphaël-Valescure", "fua:<city>"),
    "osm:n2010251922": ("Saint-Étienne Châteaucreux", "fua:<city>"),
    "osm:n3069229440": ("Strasbourg", "fua:<city>"),
    "osm:n2506173917": ("Troyes", "fua:<city>"),
    "osm:n10935384072": ("Valence-Ville", "fua:<city>"),
    "osm:n1648968005": ("Valenciennes", "fua:<city>"),
    "osm:n394710073": ("Vannes", "fua:<city>"),
    # --- BE ---
    "osm:n2929614444": ("Brugge", "fua:<city>"),
    "osm:n1178257779": ("Gent-Sint-Pieters", "fua:<city>"),
    "osm:n21309047": ("Kortrijk", "fua:<city>"),
    "osm:n446059037": ("La Louvière-Centre", "fua:<city>"),
    "osm:n7261826908": ("Mechelen-Nekkerspoel", "fua:<city>"),
    "osm:n1027979508": ("Oostende", "fua:<city>"),
    "osm:n26446051": ("Verviers-Central", "fua:<city>"),
    # --- NL ---
    "osm:n4085675596": ("'s-Hertogenbosch", "fua:<city>"),
    "osm:n4085675598": ("Alkmaar", "fua:<city>"),
    "osm:n4530820010": ("Almelo", "fua:<city>"),
    "osm:n4555468696": ("Almere Centrum", "fua:<city>"),
    "osm:n7606786768": ("Assen", "fua:<city>"),
    "osm:n43174364": ("Breda", "fua:<city>"),
    "osm:n4425618606": ("Dordrecht", "fua:<city>"),
    "osm:n4487559980": ("Enschede", "fua:<city>"),
    "osm:n1112410297": ("Groningen", "fua:<city>"),
    "osm:n5252716645": ("Haarlem", "fua:<city>"),
    "osm:n45931397": ("Hengelo", "fua:<city>"),
    "osm:n48162487": ("Leeuwarden", "fua:<city>"),
    "osm:n9604567339": ("Leiden Centraal", "fua:<city>"),
    "osm:n46792197": ("Lelystad Centrum", "fua:<city>"),
    "osm:n5311118145": ("Maastricht", "fua:<city>"),
    "osm:n44061300": ("Nijmegen", "fua:<city>"),
    "osm:n42966392": ("Roosendaal", "fua:<city>"),
    "osm:n4041466061": ("Tilburg", "fua:<city>"),
    "osm:n4555433902": ("Venlo", "fua:<city>"),
    "osm:n4487554970": ("Zwolle", "fua:<city>"),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- BE --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n6010646002": ("Charleroi-Central", "fua:Charleroi"),
    "osm:n5645805399": ("Namur", "fua:Namur"),  # export coordinate 10 km off; corrected
    "osm:n5467988372": ("Leuven", "fua:Leuven"),
    # --- FR --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n2506241285": (
        "Paris Gare de l'Est",
        "fua:Paris",
    ),  # hidden by the 2 km check: Gare du Nord is 490 m away. Terminus of the eastern night trains; id from illustrative charge row (retired curated catalog)
    "osm:n26824135": (
        "Gare de Lyon",
        "fua:Paris",
    ),  # not in any schedule export — target-network addition; southbound terminus
    "osm:n1823210835": (
        "Paris-Bercy - Bourgogne-Pays d'Auvergne",
        "network",
    ),  # night train terminus (Intercités de nuit)
    "osm:n65331500": ("Gare Montparnasse", "network"),  # westbound terminus
    "osm:w1022039965": (
        "Aéroport Charles de Gaulle 2 TGV",
        "network:airport",
    ),  # the export had 'Aéroport CDG 1', an RER-only station; the mainline station is CDG 2 TGV
    "osm:n4290857016": (
        "Lyon Part-Dieu",
        "network",
    ),  # Lyon's main station; Perrache is in as fua:Lyon
    "osm:n5615156624": ("Bordeaux-Saint-Jean", "fua:Bordeaux"),
    "osm:w429438363": ("Dijon-Ville", "fua:Dijon"),  # Dijon-Ville
    "osm:n8745537418": (
        "Lille-Europe",
        "network",
    ),  # HSL station; Lille-Flandres is in as fua:Lille. Export coordinate 4.9 km off; corrected
    "osm:n3486467305": ("Saint-Quentin", "fua:Saint-Quentin"),
    "osm:n8096578751": (
        "Montpellier-Sud-de-France",
        "network",
    ),  # HSL station; Saint-Roch is qualified
    "osm:n6960862545": (
        "Saint-Pierre-des-Corps",
        "fua:Tours",
    ),  # Tours' mainline station; Tours itself is a terminus
    "osm:n4301927151": ("Morlaix", "network"),  # Brittany line to Brest
    "osm:n3916678066": (
        "Hendaye",
        "border",
    ),  # FR/ES; the Spanish side Irun may already qualify
    # --- LU --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n3964570119": (
        "Luxembourg",
        "fua:Luxembourg",
    ),  # export coordinate 31 km off (README); corrected
    # --- NL --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n7510315998": ("Eindhoven Centraal", "fua:Eindhoven"),
    "osm:n2235186029": ("Apeldoorn", "fua:Apeldoorn"),
}

In [4]:
# Iberia
ADDITIONS_IBERIA = {
    # --- ES ---
    "osm:n13782316672": ("A Coruña", "fua:<city>"),
    "osm:w28776478": ("Abando Indalecio Prieto", "fua:<city>"),
    "osm:n10914769161": ("Alacant Terminal", "fua:<city>"),
    "osm:n11016395831": ("Albacete Los Llanos", "fua:<city>"),
    "osm:n2182333421": ("Algeciras-Paco de Lucía", "fua:<city>"),
    "osm:n30546837": ("Avilés", "fua:<city>"),
    "osm:n2962633346": ("Badajoz", "fua:<city>"),
    "osm:n617134268": ("Cartagena", "fua:<city>"),
    "osm:n13894649638": ("Castelló", "fua:<city>"),
    "osm:n13717016710": ("Ciudad Real", "fua:<city>"),
    "osm:n13714976346": ("Cáceres", "fua:<city>"),
    "osm:n259625422": ("Cádiz", "fua:<city>"),
    "osm:n7567516121": ("Córdoba Julio Anguita", "fua:<city>"),
    "osm:n1738646773": ("El Puerto de Santa María", "fua:<city>"),
    "osm:w24930154": ("Ferrol", "fua:<city>"),
    "osm:n7201979115": ("Granada", "fua:<city>"),
    "osm:n6313228293": ("Guadalajara", "fua:<city>"),
    "osm:n5580567331": ("Huelva", "fua:<city>"),
    "osm:n8051540821": ("Huesca", "fua:<city>"),
    "osm:n7747872372": ("Huércal-Viator", "fua:<city>"),
    "osm:n5299078295": ("León", "fua:<city>"),
    "osm:n2459459539": ("Logroño", "fua:<city>"),
    "osm:w86123754": ("Lugo", "fua:<city>"),
    "osm:n13017334754": ("Murcia del Carmen", "fua:<city>"),
    "osm:n2609534280": ("Málaga María Zambrano", "fua:<city>"),
    "osm:n2039781019": ("Mérida", "fua:<city>"),
    "osm:n1842017741": ("Ourense-Empalme", "fua:<city>"),
    "osm:n4586092220": ("Oviedo / Uviéu", "fua:<city>"),
    "osm:n1939943099": ("Palencia", "fua:<city>"),
    "osm:n11757382798": ("Pamplona / Iruña", "fua:<city>"),
    "osm:n1069592576": ("Ponferrada", "fua:<city>"),
    "osm:n453651815": ("Pontevedra", "fua:<city>"),
    "osm:n4448930346": ("Salamanca", "fua:<city>"),
    "osm:n2340360836": ("San Fernando-Bahía Sur", "fua:<city>"),
    "osm:n5893228216": ("Santander", "fua:<city>"),
    "osm:n12768629140": ("Santiago de Compostela - Daniel Castelao", "fua:<city>"),
    "osm:n191262271": ("Sevilla - Santa Justa", "fua:<city>"),
    "osm:n2328131203": ("Talavera de la Reina", "fua:<city>"),
    "osm:n13894696022": ("Tarragona", "fua:<city>"),
    "osm:n7246471728": ("València - Estació del Nord", "fua:<city>"),
    "osm:n7246471727": ("València Joaquín Sorolla", "fua:<city>"),
    "osm:n12819737430": ("Vigo-Urzáiz", "fua:<city>"),
    "osm:n29568804": ("Vitoria-Gasteiz", "fua:<city>"),
    "osm:n791063229": ("Zamora", "fua:<city>"),
    # --- PT ---
    "osm:n10783341030": ("Aveiro", "fua:<city>"),
    "osm:n1393070418": ("Barroselas", "fua:<city>"),
    "osm:n10783341036": ("Braga", "fua:<city>"),
    "osm:n10783341029": ("Coimbra-B", "fua:<city>"),
    "osm:n46756926": ("Faro", "fua:<city>"),
    "osm:n10783341033": ("Gaia", "fua:<city>"),
    "osm:n3941781457": ("Guimarães", "fua:<city>"),
    "osm:n6297158592": ("Lisboa - Oriente", "fua:<city>"),
    "osm:n10783341023": ("Porto - Campanhã", "fua:<city>"),
    "osm:n2861321998": ("Viana do Castelo", "fua:<city>"),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- ES --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n2158044594": ("Barcelona - Sants", "fua:Barcelona"),
    "osm:n7499028216": (
        "Madrid-Puerta de Atocha-Almudena Grandes",
        "fua:Madrid",
    ),  # removed 2026-08-24 as duplicate of step 5; step 5 lost it 2026-08-28. Take the mainline object, not Atocha-Cercanías. The resolver's name search lands on Atocha-Cercanías (the object the August note warned about) — id pinned to the mainline node
    "osm:n9821007500": (
        "Madrid-Chamartín-Clara Campoamor",
        "network",
    ),  # northbound/international terminus; Madrid FUA is Atocha
    "osm:n7490586752": ("Girona", "fua:Girona"),
    "osm:n7512362062": (
        "Figueres",
        "border",
    ),  # break-of-gauge interchange; Figueres-Vilafant is the HSL station 2 km west
    "osm:n7487337013": ("Lleida-Pirineus", "fua:Lleida"),
    "osm:n4813092198": ("Zaragoza-Delicias", "fua:Zaragoza"),
    "osm:n13021298528": ("Burgos - Rosa Manzano", "fua:Burgos"),
    "osm:n9175580427": ("Valladolid - Campo Grande", "fua:Valladolid"),
}

In [5]:
# Italy
ADDITIONS_ITALY = {
    # ONTD has no Roma Ostiense (the station is an OSM relation, invisible to
    # the node-only ONTD export); without this carry, step 5 used to fall back
    # to the Piramide metro object 550 m away.
    # --- IT ---
    "osm:r1821284": (
        "Roma Ostiense",
        "night_train_stop",
    ),  # ONTD has only the adjacent Metro B object; station is a relation
    "osm:n5324492776": ("Acireale", "fua:<city>"),
    "osm:n12291596709": ("Alessandria", "fua:<city>"),
    "osm:n7460292092": ("Ancona", "fua:<city>"),
    "osm:n593748428": ("Arezzo Pescaiola", "fua:<city>"),
    "osm:n8820637017": ("Avellino", "fua:<city>"),
    "osm:n1699232800": ("Barletta", "fua:<city>"),
    "osm:n8607336257": ("Bergamo", "fua:<city>"),
    "osm:n7473059189": ("Campobasso", "fua:<city>"),
    "osm:n1215079672": ("Caserta", "fua:<city>"),
    "osm:n603353943": ("Cefalù", "tourism:<region>"),
    "osm:n258613100": ("Cerignola Campagna", "fua:<city>"),
    "osm:n1279764780": ("Cosenza Vaglio Lise", "fua:<city>"),
    "osm:n2121919441": ("Ferrara", "fua:<city>"),
    "osm:n738159083": ("L'Aquila", "fua:<city>"),
    "osm:n7042610811": ("Milazzo", "fua:<city>"),
    "osm:n726611782": ("Modena", "fua:<city>"),
    "osm:n11802851419": ("Novara", "fua:<city>"),
    "osm:n5836604869": ("Parma", "fua:<city>"),
    "osm:n211030020": ("Pavia", "fua:<city>"),
    "osm:n249236145": ("Perugia", "fua:<city>"),
    "osm:n1862274592": ("Pesaro", "fua:<city>"),
    "osm:n267591085": ("Pescara Centrale", "fua:<city>"),
    "osm:n13742535643": ("Piacenza", "fua:<city>"),
    "osm:n842367835": ("Potenza Centrale", "fua:<city>"),
    "osm:n9067692438": ("Ravenna", "fua:<city>"),
    "osm:n82549162": ("Reggio Emilia", "fua:<city>"),
    "osm:n1274172585": ("Sant'Agata di Militello", "tourism:<region>"),
    "osm:n1764381735": ("Trento", "fua:<city>"),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- IT --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n7472007553": ("Levanto", "tourism:Cinque Terre"),
    "osm:n2052772794": ("Ventimiglia", "border"),  # FR/IT
    "osm:n169701311": (
        "Milano Lambrate",
        "network",
    ),  # second Milano station on the eastern side; Centrale is qualified
    "osm:n9635608905": (
        "Milano Rogoredo",
        "network",
    ),  # southern Milano stop of Genova/Bologna trains
    "osm:n279371582": (
        "Firenze Castello",
        "network",
    ),  # bypass station used by through night trains avoiding SMN
    "osm:n6063641896": ("Padova", "fua:Padova"),
    "osm:n6061881431": ("Vicenza", "fua:Vicenza"),
    "osm:n4553822557": ("Brescia", "fua:Brescia"),
    "osm:n8416079151": (
        "Vercelli",
        "fua:Vercelli",
    ),  # export coordinate was Grosseto — wrong; corrected
    "osm:n12299676794": (
        "Savona",
        "fua:Savona",
    ),  # export coordinate was Aarau — wrong; corrected
    "osm:n4427972085": (
        "Imperia",
        "tourism:Riviera dei Fiori",
    ),  # Imperia Oneglia (the export's stop) closed in 2016 when the new through station Imperia opened 1.2 km west — the new one is the stop
    "osm:n12048036719": ("Orte", "network"),  # Roma–Firenze/Ancona junction
    "osm:n6813784587": (
        "Calalzo - Pieve di Cadore - Cortina",
        "tourism:Cortina d'Ampezzo",
    ),
}

In [6]:
# United Kingdom, Ireland
ADDITIONS_UNITED_KINGDOM = {
    # --- GB ---
    "osm:n7998566986": ("Ashford International", "fua:<city>"),
    "osm:n4461326005": ("Bangor", "fua:<city>"),
    "osm:n12248421687": ("Belfast Grand Central", "fua:<city>"),
    "osm:n6765532062": ("Birmingham New Street", "fua:<city>"),
    "osm:n6634567434": ("Bournemouth", "fua:<city>"),
    "osm:n7209380367": ("Bradford Interchange", "fua:<city>"),
    "osm:n20947173": ("Brighton", "fua:<city>"),
    "osm:n7167271113": ("Bristol Temple Meads", "fua:<city>"),
    "osm:n573566827": ("Cambridge", "fua:<city>"),
    "osm:n3453612249": ("Canterbury West", "fua:<city>"),
    "osm:n6605149666": ("Cardiff Central", "fua:<city>"),
    "osm:n5028607042": ("Chester", "fua:<city>"),
    "osm:n6688385690": ("Dover Priory", "ferry:<port>"),
    "osm:n6013523209": ("Exeter St Davids", "fua:<city>"),
    "osm:n6646199707": ("Gloucester", "fua:<city>"),
    "osm:n7154209250": ("Holyhead", "ferry:<port>"),
    "osm:n6012826246": ("Hull Paragon Interchange", "ferry:<port>"),
    "osm:n7156706693": ("Leeds", "fua:<city>"),
    "osm:n4292139459": ("Leicester", "fua:<city>"),
    "osm:n6960405293": ("Liverpool Lime Street", "fua:<city>"),
    "osm:n5064005964": ("Manchester Piccadilly", "fua:<city>"),
    "osm:n7159380475": ("Milton Keynes Central", "fua:<city>"),
    "osm:n195885858": ("Newcastle", "fua:<city>"),
    "osm:n7158616254": ("Norwich", "fua:<city>"),
    "osm:n324650068": ("Nottingham", "fua:<city>"),
    "osm:n6481707942": ("Oxford", "fua:<city>"),
    "osm:n2612643529": ("Peterborough", "fua:<city>"),
    "osm:n6010790017": ("Portsmouth and Southsea", "ferry:<port>"),
    "osm:n5784212748": ("Sheffield", "fua:<city>"),
    "osm:n638908005": ("Southampton Central", "fua:<city>"),
    "osm:n7140234411": ("Southend Victoria", "fua:<city>"),
    "osm:n6900337987": ("Swansea", "fua:<city>"),
    "osm:n104734": ("Swindon", "fua:<city>"),
    "osm:n6634567442": ("Wolverhampton", "fua:<city>"),
    "osm:n7989407332": ("Wrexham General", "fua:<city>"),
    # --- IE ---
    "osm:n5355226792": ("Cork Kent", "fua:<city>"),
    "osm:n7198337980": ("Dublin Connolly", "fua:<city>"),
    "osm:n6854406241": ("Dublin Heuston", "fua:<city>"),
    "osm:n6854415949": ("Galway Ceannt", "fua:<city>"),
    "osm:n6740364330": ("Limerick Colbert", "fua:<city>"),
    "osm:n6852246817": ("Waterford Plunkett", "fua:<city>"),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- GB --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n7159162454": (
        "London King's Cross",
        "network",
    ),  # removed 2026-08-24 as duplicate of step 5; step 5 lost it 2026-08-28. Caledonian Sleeper terminus is Euston; King's Cross is the ECML terminus
    "osm:n3662847634": (
        "London St. Pancras International",
        "network",
    ),  # Eurostar terminus — any continental night train into London ends here
}

In [7]:
# Nordics
ADDITIONS_NORDICS = {
    # --- SE ---
    "osm:n7135739559": ("Borås C", "fua:<city>"),
    "osm:r10274650": ("Härnösand resecentrum", "fua:<city>"),
    "osm:w155603926": ("Station Åre", "night_train_stop"),
    # --- DK ---
    "osm:n3419486092": ("Aalborg", "fua:<city>"),
    "osm:n5026479524": ("Aarhus H", "fua:<city>"),
    "osm:n1655765253": ("Hirtshals", "ferry:<port>"),
    # --- FI ---
    "osm:n1716259527": ("Espoo", "fua:<city>"),
    "osm:n259004650": ("Jyväskylä", "fua:<city>"),
    "osm:n603918145": ("Kotka satama", "network"),
    "osm:n292809487": ("Kuopio", "fua:<city>"),
    "osm:n537913195": ("Lahti", "fua:<city>"),
    "osm:n340019021": ("Tikkurila", "fua:<city>"),
    "osm:n91925127": ("Vaasa", "network"),
    "osm:n4993961319": (
        "Esbjerg",
        "ferry:Esbjerg — carried over from the curated catalog, which step 7 now replaces",
    ),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- DK --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n3739700410": (
        "Københavns Hovedbanegård",
        "fua:København",
    ),  # the main station; only Syd and Kastrup were in. Also carried an illustrative charge row that step 10 now reports stale; id from tests/conftest.py STOPS_COPENHAGEN_STOCKHOLM and the illustrative charge row
    "osm:n7895320358": (
        "Høje Taastrup",
        "network",
    ),  # western Copenhagen stop of trains to Germany
    "osm:n5643031046": (
        "Køge Nord",
        "network",
    ),  # HSL station on the Copenhagen–Ringsted line
    "osm:n2322481507": ("Ringsted", "network"),  # Sjælland junction
    "osm:n2323484867": ("Slagelse", "network"),  # Copenhagen–Fyn line
    "osm:n4177059309": ("Nykøbing F", "network"),  # Fehmarnbelt corridor from ~2029
    # --- NO --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n5526332038": (
        "Narvik",
        "network",
    ),  # Ofotbanen terminus of the Stockholm night train
    "osm:n5526331839": ("Fredrikstad", "fua:Fredrikstad"),
    "osm:n2408227293": ("Sarpsborg", "fua:Sarpsborg"),
    "osm:n5720557915": ("Moss", "fua:Moss"),  # export coordinate 2.5 km off; corrected
    "osm:n5720557883": ("Halden", "border"),  # NO/SE, Østfoldbanen
    # --- SE --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n1305188216": (
        "Göteborgs central",
        "fua:Göteborg",
    ),  # the export carried this twice (Göteborg C / Göteborg Centralen station) — one entry
    "osm:n8137601572": ("Örebro centralstation", "fua:Örebro"),
    "osm:n10381569083": ("Västerås Central", "fua:Västerås"),
    "osm:n3158244405": (
        "Katrineholm C",
        "network",
    ),  # Stockholm–Göteborg/Malmö junction
    "osm:n11383125342": (
        "Kiruna",
        "tourism:Lapland",
    ),  # Stockholm–Narvik night train destination (was in the frozen export)
    "osm:n4067854164": ("Gällivare", "network"),  # Malmbanan; night train calls
}

In [8]:
# Central Europe
ADDITIONS_CENTRAL_EUROPE = {
    # --- PL ---
    "osm:n413346673": ("Elbląg", "fua:<city>"),
    "osm:n2050000245": ("Gorzów Wielkopolski", "fua:<city>"),
    "osm:n3831584572": ("Grudziądz", "fua:<city>"),
    "osm:n3357286930": ("Głogów", "fua:<city>"),
    "osm:n811380395": ("Inowrocław", "fua:<city>"),
    "osm:n3459064660": ("Kalisz", "fua:<city>"),
    "osm:n29830753": ("Konin", "fua:<city>"),
    "osm:n2627870779": ("Legnica", "fua:<city>"),
    "osm:n5356872372": ("Lubin", "fua:<city>"),
    "osm:n475592274": ("Nowy Sącz", "fua:<city>"),
    "osm:n131851982": ("Olsztyn Główny", "fua:<city>"),
    "osm:n528526130": ("Ostrów Wielkopolski", "fua:<city>"),
    "osm:n1864585459": ("Piotrków Trybunalski", "fua:<city>"),
    "osm:n3459469757": ("Piła Główna", "fua:<city>"),
    "osm:n1991734621": ("Płock", "fua:<city>"),
    "osm:n842079165": ("Wałbrzych Miasto", "fua:<city>"),
    "osm:n3459316528": ("Włocławek", "fua:<city>"),
    "osm:n372254443": ("Zamość", "fua:<city>"),
    "osm:n1992495561": ("Łomża", "fua:<city>"),
    # --- CZ ---
    "osm:n3279883031": ("Hradec Králové hlavní nádraží", "fua:<city>"),
    "osm:n5648124921": ("Most", "fua:<city>"),
    "osm:n30077682": ("Plzeň hlavní nádraží", "fua:<city>"),
    # --- SK ---
    "osm:n8012637130": ("Banská Bystrica", "fua:<city>"),
    "osm:n7066101885": ("Nitra", "fua:<city>"),
    "osm:n346420319": ("Prešov", "fua:<city>"),
    "osm:n10607895201": ("Zvolen nákladná stanica", "network"),
    "osm:n6446509215": ("Žilina", "fua:<city>"),
    # --- HU ---
    "osm:n268213797": ("Miskolc-Tiszai", "fua:<city>"),
    "osm:n25546152": ("Pécs", "fua:<city>"),
    "osm:n93800956": ("Szombathely", "fua:<city>"),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- CZ --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n32522686": ("České Budějovice", "fua:České Budějovice"),
    # --- HU --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n748766808": (
        "Budapest-Déli",
        "network",
    ),  # removed 2026-08-24 as duplicate of step 5; step 5 lost it 2026-08-28. Terminus toward Balaton/Croatia
    "osm:n4214909592": ("Kecskemét", "fua:Kecskemét"),
    "osm:n5991440733": (
        "Szeged",
        "fua:Szeged",
    ),  # the export had Szeged-Kiskundorozsma (freight/junction); the main station is missing entirely
    # --- PL --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n3298971170": (
        "Białystok",
        "fua:Białystok",
    ),  # the export had Białystok Starosielce (suburban); the main station is missing entirely. Rail Baltica corridor
    "osm:n3067068079": ("Suwałki", "network"),  # Rail Baltica corridor PL/LT
    "osm:n367993397": ("Ełk", "network"),  # Rail Baltica corridor; junction
    "osm:n3416823373": (
        "Warszawa Gdańska",
        "network",
    ),  # used by trains bypassing Centralna
    "osm:n3258261975": ("Rzepin", "border"),  # DE/PL, Berlin–Poznań
    "osm:n3525511368": ("Zebrzydowice", "border"),  # PL/CZ
    "osm:n529102526": (
        "Oświęcim",
        "network",
    ),  # Kraków–Czechowice line; memorial-site visitors
    # --- SK --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n30073060": (
        "Bratislava-Petržalka",
        "network",
    ),  # Wien-side station of Bratislava
}

In [9]:
# Baltics
ADDITIONS_BALTICS = {
    # --- EE ---
    "osm:n529932898": ("Narva", "fua:<city>"),
    "osm:n30402685": ("Paldiski", "fua:<city>"),
    "osm:n8761131036": ("Tartu", "fua:<city>"),
    # --- LV ---
    "osm:n7800844381": ("Daugavpils", "fua:<city>"),
    "osm:n7799314251": ("Liepāja", "fua:<city>"),
    "osm:n252636397": ("Ventspils-1", "ferry:<port>"),
    # --- LT ---
    "osm:n6629286841": (
        "Kaunas",
        "network:break-of-gauge — 1435 Rail Baltica meets the 1520 legacy "
        "network; LT's second city. Was a step 5 stop under the frozen "
        "schedule export until no active train called there (2026-08-29)",
    ),
    "osm:n99172829": ("Klaipėda", "fua:<city>"),
    "osm:n6624987344": ("Vilnius", "fua:<city>"),
    "osm:n1370770391": ("Šiauliai", "fua:<city>"),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- EE --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n650927310": ("Tallinn", "fua:Tallinn"),  # today's main station (1520 mm)
    "osm:n27508651": (
        "Ülemiste",
        "network:Rail Baltica terminal",
        "infra-2032",
    ),  # existing Elron stop being rebuilt as the Rail Baltica terminal; as a night-train stop it is 2032
    # --- LT --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n5753746373": ("Panevėžys", "fua:Panevėžys"),  # Rail Baltica corridor
    "osm:n360353202": (
        "Marijampolė",
        "network",
    ),  # Rail Baltica corridor PL/LT; existing 1435 mm station
    # --- LV --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n7799405223": (
        "Rīgas Centrālā stacija",
        "fua:Rīga",
    ),  # Rīgas Centrālā stacija; the heavy_rail node, not the station-building way beside it
}

In [10]:
# South-eastern Europe
ADDITIONS_SOUTH_EASTERN_EUROPE = {
    # --- SI ---
    "osm:n283207146": ("Bled Jezero", "fua:<city>"),
    "osm:n270129111": ("Postojna", "tourism:<region>"),
    # --- HR ---
    "osm:n5668860678": ("Pula", "tourism:<region>"),
    # --- BA ---
    "osm:n2038791178": ("Doboj", "network"),
    "osm:n312280159": ("Mostar", "fua:<city>"),
    "osm:n942406094": ("Sarajevo", "fua:<city>"),
    "osm:n10212563520": ("Zenica", "fua:<city>"),
    # --- RS ---
    "osm:n1601596124": ("Ниш", "fua:<city>"),
    "osm:n893439322": ("Суботица", "fua:<city>"),
    # --- ME ---
    "osm:n10728069934": ("Nikšić", "fua:<city>"),
    # --- MK ---
    "osm:n3407148607": ("Битола", "network"),
    "osm:n1635043504": ("Велес", "fua:<city>"),
    "osm:n134701987": ("Гевгелија", "border"),
    "osm:n408145528": ("Куманово", "fua:<city>"),
    "osm:n9947846021": ("Скопје", "fua:<city>"),
    # --- AL ---
    "osm:n13895194677": ("Durrës", "fua:<city>"),
    # Tiranë has NO rail station to add: the city's terminus was demolished
    # in 2013 and the line now ends at Kashar. The entry that stood here
    # (osm:n13895194676, "Terminali i Transportit Publik Tiranë") was the
    # BUS terminal — it seeded with gauges_mm NULL, became Albania's
    # centroid-nearest reference station, and silently cost the country all
    # 19 of its rows in the relations matrix (2026-08-29). Removed; re-add
    # a real station if the Tiranë–Durrës rebuild ever reaches OSM.
    # --- XK ---
    "osm:n2107256271": ("Ferizaj", "fua:<city>"),
    "osm:n1613271652": ("Prishtinë", "fua:<city>"),
    # --- BG ---
    "osm:n14055863804": ("Сливен", "night_train_stop"),
    "osm:n1243304538": ("Botoșani", "fua:<city>"),
    "osm:n9244693114": ("Brăila", "fua:<city>"),
    "osm:n487098710": ("Călărași Sud", "fua:<city>"),
    # Dej Călători (osm:n2235196597) stood here from 2026-08-29 because step 5
    # had fuzzy-matched ONTD Dej to the town's BUS terminal ("Autogara Dej",
    # excluded in step 10). Since step 4 tops the register up from ONTD, step
    # 5 qualifies Dej Călători itself (exact match) — so the line is gone,
    # or guard 2 raises on it (2026-09-01).
    "osm:w272187224": ("Dej Triaj", "night_train_stop"),
    "osm:n8176378047": ("Galați", "fua:<city>"),
    "osm:n303278705": ("Piatra-Neamț", "fua:<city>"),
    "osm:n8607130227": ("Pitești", "fua:<city>"),
    "osm:n516125655": ("Râmnicu Vâlcea", "fua:<city>"),
    "osm:n536752658": ("Slatina", "fua:<city>"),
    "osm:n7905078406": ("Tulcea Oraș", "fua:<city>"),
    "osm:n9554314257": ("Târgoviște", "fua:<city>"),
    "osm:n258668249": ("Târgu Mureș", "fua:<city>"),
    # --- GR ---
    "osm:n9643537166": ("Βόλος", "fua:<city>"),
    "osm:n4925130296": ("Θεσσαλονίκη", "fua:<city>"),
    "osm:n6175162599": ("Κατερίνη", "fua:<city>"),
    "osm:n287554537": ("Λάρισα", "fua:<city>"),
    "osm:n308841776": ("Ξάνθη", "fua:<city>"),
    "osm:n9721698903": ("Ρίο", "fua:<city>"),
    "osm:n4883927548": ("Σέρραι", "fua:<city>"),
    "osm:w609979867": ("Αθήνα", "fua:<city>"),
    "osm:n13254223965": (
        "Централна гара Бургас",
        "night_train_stop (absent from ONTD)",
    ),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- BG --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n2321100569": ("Пирдоп", "network"),  # Sofia–Burgas via Karlovo line
    # --- HR --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n10068642643": ("Slavonski Brod", "network"),  # corridor X
    "osm:n2445033209": ("Vukovar", "fua:Vukovar"),
    # --- RO --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n13247684141": ("București Băneasa", "network"),  # northern București station
    "osm:n11761562381": ("Vișeu de Jos", "network"),  # Maramureș line toward Sighetu
    "osm:n2270205803": (
        "Sighetu Marmației",
        "border",
    ),  # RO/UA; Solotvyno is 1.7 km away across the Tisza but a different country
    # --- RS --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n2966093688": ("Нови Сад", "fua:Novi Sad"),
}

In [11]:
# Eastern Europe, Türkiye
ADDITIONS_EASTERN_EUROPE = {
    # --- UA ---
    "osm:n9708140929": ("Алчевськ", "fua:<city>"),
    "osm:n3815614789": ("Бердянськ (експ.)", "fua:<city>"),
    "osm:n9695166282": ("Дебальцеве", "fua:<city>"),
    "osm:n278290881": ("Донецьк", "fua:<city>"),
    "osm:n10175634575": ("Луганськ", "fua:<city>"),
    "osm:n1201406760": ("Маріуполь", "fua:<city>"),
    "osm:n652003827": ("Мелітополь", "fua:<city>"),
    "osm:n2635702849": (
        "Миколаїв",
        "night_train_stop (step 6 had Миколаїв-Вантажний, a freight station)",
    ),
    "osm:n8220188327": ("Росинка", "fua:<city>"),
    # Crimean stations (Севастополь osm:n305264647, Симферополь
    # osm:n3693422903, Евпатория-Курорт osm:n676673228, Керчь-Порт
    # osm:n2469903575) are removed, loudly: the peninsula's rail network is
    # a DISCONNECTED component in the routing graph — the Perekop and
    # Chonhar lines to Kherson are not routable (OSM tagging since 2022) —
    # so any proposal touching one of these stops can only ever answer
    # routing_error. Real stations, deliberately not modelled while
    # unreachable; re-add when the mainland links return to OSM.
    "osm:n11742271914": ("Херсон", "fua:<city>"),
    "osm:n4237097872": ("Шепетівка", "fua:<city>"),
    # --- TR ---
    "osm:n2726063373": ("Adana", "fua:<city>"),
    "osm:n1033799242": ("Denizli", "fua:<city>"),
    "osm:w88493479": ("Gaziantep Garı", "fua:<city>"),
    "osm:n1035592982": ("Isparta", "fua:<city>"),
    "osm:n11082593543": ("Kırkikievler", "fua:<city>"),
    "osm:n2596542075": ("Mersin Garı", "fua:<city>"),
    "osm:n1550726413": ("Muş", "fua:<city>"),
    "osm:n125326513": ("Osmaniye", "fua:<city>"),
    "osm:n1023854236": ("Samsun", "tourism:<region>"),
    "osm:n13480008012": ("Zonguldak", "tourism:<region>"),
    "osm:n719596870": (
        "Полтава-Київська",
        "night_train_stop (absent from ONTD) — 19 trips",
    ),
    "osm:n1276842137": (
        "Кривий Ріг-Головний",
        "night_train_stop (replaces Кривий Ріг, a smaller station 3.8 km off)",
    ),
    # ---- 2026-09 gap closure: stations the live catalog had lost, resolved
    #      by step6a_resolve_candidates.py from step6_gap_closure_2026-09.csv ----
    # --- UA --- (step 6a, step6_gap_closure_2026-09.csv)
    "osm:n1408518014": (
        "Коростень",
        "network",
    ),  # Kyiv–Kovel/Warszawa junction; search in Cyrillic
    "osm:n13072326058": ("Краматорськ", "fua:Kramatorsk"),
    "osm:n1065061073": ("Слов'янськ", "fua:Sloviansk"),
    "osm:n610644672": ("Ізюм", "network"),
    "osm:n1604695980": (
        "Шептицький",
        "fua:Sheptytskyi",
    ),  # the city was renamed Шептицький in 2024 and OSM follows; the export still says Chervonohrad
    "osm:w803062227": ("Дубно", "network"),
    "osm:n783848373": (
        "Умань",
        "fua:Uman",
    ),  # the export coordinate is Uman's BUS station; the railway station Умань is 3.4 km away — id pinned
}

## Removed 2026-08-24 — duplicates of step 5

Step 6 was selected against a step 5 run that silently dropped 44 % of the
network, so several picks compensated for stops that are back, and a few
picked a metro/S-Bahn object sitting metres from the mainline station. The
guards below now block this class of entry; these were removed when the
guards were introduced (qualified counterpart in parentheses):

*Same OSM object, now qualified by step 5:* Remscheid-Lennep,
Rimini Torre Pedrera, Schönenwerd, Vignale-Riotorto, Централна гара Русе,
Централна гара София.

*Second OSM object for an already-qualified station:* Gesundbrunnen
(Berlin Gesundbrunnen), Spandau (Berlin-Spandau), Südkreuz (Berlin Südkreuz),
Euston (London Euston), King's Cross St Pancras (London King's Cross),
Gare du Midi (Bruxelles-Midi), Diamant (Antwerpen-Centraal),
Atocha-Cercanías (Madrid-Puerta de Atocha), Déli pályaudvar (Budapest-Déli),
Kelenföld vasútállomás (Budapest-Kelenföld), Lugano funicolare (Lugano),
Arlanda central (Arlanda norra), Вокзальна (Київ-Пасажирський).

*Not railway stations at all:* Vörösmarty utca (Budapest M1 metro; the FUA is
covered by Budapest-Nyugati), Hauptbahnhof Arnulf-Klett-Platz (Stuttgart
Stadtbahn; Stuttgart Hauptbahnhof is qualified 300 m away).

*Objects replaced with the mainline station:* Aarhus H (light-rail node →
`osm:n5026479524`), Athens (metro Σταθμός Λαρίσης → railway station
`osm:w609979867`).

**Addendum 2026-09-01.** Step 5 went live against the ONTD workbook on
2026-08-28 and no longer qualifies several of the counterparts above —
Berlin Gesundbrunnen, Berlin-Spandau, London King's Cross, Madrid Atocha,
Budapest-Déli among them — so those stations fell out of the catalog
entirely. They are back as step 6 additions in the 2026-09 gap closure
below. The lesson is recorded in step 10, which now reports every stop the
previous catalog had and the new one lacks (`data/step10_dropped_stops.csv`):
a prune of step 6 against step 5 is only as durable as step 5's membership.

## Gap closure 2026-09

`step6_gap_closure_2026-09.csv` (tracked, next to this notebook) is the
record of the batch that closed the gap the addendum describes: every
station the frozen schedule export contained that had no catalog stop
within 2 km, plus the big-city second stations that check hides (Paris
Est/Lyon/Bercy/Montparnasse, Frankfurt Hbf, London St Pancras, Köln
Messe/Deutz) — with a reason and `infra_versions` each, and the stations
*not* added listed with why. **126 of them are in the dicts above**, under
the `# ---- 2026-09 gap closure` comment in each region, resolved by

```
uv run python step6a_resolve_candidates.py step6_gap_closure_2026-09.csv
```

and pasted as printed. 32 rows are `skip` (covered under another object
or name, wrong export coordinate, not a station), 3 are `review`, and 2
are `defer` — Rīga Airport and Pärnu International, Rail Baltica stations
that are `railway=construction` in OSM and so not in step 3b; re-run the
resolver when they appear and they print as `infra-2032` lines. The same
command is how the next batch gets in: copy the CSV, keep the header,
add rows, run, paste.


## Combine, resolve and write

In [ ]:
ADDITIONS = {}
for group in (
    ADDITIONS_GERMANY,
    ADDITIONS_FRANCE,
    ADDITIONS_IBERIA,
    ADDITIONS_ITALY,
    ADDITIONS_UNITED_KINGDOM,
    ADDITIONS_NORDICS,
    ADDITIONS_CENTRAL_EUROPE,
    ADDITIONS_BALTICS,
    ADDITIONS_SOUTH_EASTERN_EUROPE,
    ADDITIONS_EASTERN_EUROPE,
):
    overlap = ADDITIONS.keys() & group.keys()
    if overlap:
        raise ValueError(f"stop listed in two regions: {sorted(overlap)}")
    for stop_id, entry in group.items():
        # (name, reason) or (name, reason, infra_versions) — normalised to
        # three elements here so the guards below need only one shape.
        if len(entry) == 2:
            label, reason = entry
            infra = INFRA_BOTH
        elif len(entry) == 3:
            label, reason, infra = entry
        else:
            raise ValueError(
                f"{stop_id}: entry must have 2 or 3 elements, got {entry!r}"
            )
        parts = {p.strip() for p in infra.replace(",", ";").split(";") if p.strip()}
        if not parts or parts - INFRA_VALID:
            raise ValueError(
                f"{stop_id} ({label}): infra_versions {infra!r} — use INFRA_BOTH, "
                f"INFRA_2026 or INFRA_2032"
            )
        ADDITIONS[stop_id] = (label, reason, ";".join(sorted(parts)))

print(f"manual additions: {len(ADDITIONS)}")
print(
    "by infra_versions:",
    Counter(infra for _, _, infra in ADDITIONS.values()).most_common(),
)

In [ ]:
# Name and coordinates come from step 3b. Country prefers ONTD via step 4
# (curated national data); step 3b's own country column is only ~1% populated,
# so the handful of stops ONTD doesn't cover are listed explicitly below rather
# than pulled from the legacy step 6 file — a dozen values are not worth a file
# dependency, and here they are visible and reviewable.
MANUAL_COUNTRY = {
    "osm:n14055863804": "BG",
    "osm:n13254223965": "BG",
    "osm:w28776478": "ES",
    "osm:w24930154": "ES",
    "osm:w86123754": "ES",
    "osm:w112095036": "FR",
    "osm:w272187224": "RO",
    "osm:r10274650": "SE",
    "osm:n1550726413": "TR",
    "osm:n4993961319": "DK",
    "osm:w609979867": "GR",
    "osm:r1821284": "IT",
    # 2026-09 gap closure — ways/nodes the ONTD register has no row for
    "osm:w1022039965": "FR",  # Aéroport Charles de Gaulle 2 TGV
    "osm:w429438363": "FR",  # Dijon-Ville
    "osm:n7799405223": "LV",  # Rīgas Centrālā stacija
    "osm:w803062227": "UA",  # Дубно
}

osm = {}
with open(
    ensure_local("step3b_output_osm_stations_classified.csv"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        osm[row["stop_id"]] = row

ontd_country = {}
ontd_of_osm = {}
with open(
    local_input("step4_MatchingONTDtoOSM.csv", "step4_MatchingONTDtoOSM.ipynb"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        stop_id = (row.get("osm_stop_id") or "").strip()
        if not stop_id:
            continue
        country = (row.get("ontd_country") or "").strip().upper()
        if country:
            ontd_country.setdefault(stop_id, country)
        ontd_of_osm.setdefault(stop_id, row["ontd_id"])

unknown = sorted(set(ADDITIONS) - set(osm))
if unknown:
    raise KeyError(
        f"{len(unknown)} stop id(s) not in step 3b — typo, or the OSM extract was "
        f"refreshed and the object is gone: {unknown[:10]}"
    )

# --- guard 1: the picked object has to be a railway station -----------------
# Step 3b classifies but drops nothing, so metro, tram, bus and funicular
# objects sit in the same lookup — and metres from a mainline station they
# carry almost the same name. A ferry pier classifies as "other" and is
# legitimate, but only where the reason says the stop is there for the ferry.
wrong_mode = []
for stop_id, (label, reason, _infra) in ADDITIONS.items():
    mode = osm[stop_id]["station_mode"]
    ferry_pick = osm[stop_id]["mode_rule"] == "ferry_terminal" and reason.startswith(
        "ferry"
    )
    if mode == "urban_transit" or (mode == "other" and not ferry_pick):
        wrong_mode.append((label, stop_id, mode, osm[stop_id]["mode_rule"]))
if wrong_mode:
    for label, stop_id, mode, rule in wrong_mode:
        print(f"  {label[:40]:42} {stop_id:22} {mode} ({rule})")
    raise ValueError(
        f"{len(wrong_mode)} addition(s) point at a metro/tram/bus/funicular "
        "object, not the railway station — replace the id with the mainline "
        "station's (step 3b usually has it within 100 m of the picked object)"
    )

# --- guard 2: no addition may duplicate a step 5 stop -----------------------
# Step 10 dedups on OSM id alone, so a station qualifying through *different*
# OSM objects in the two layers would be written twice — the way the catalog
# once carried Gesundbrunnen beside Berlin Gesundbrunnen. Three tests, each an
# error: same OSM id, same ONTD station via the step 4 join, or within
# SAME_STATION_KM of a qualified stop (different objects for one station sit
# metres apart; distinct stations in one city do not).
SAME_STATION_KM = 0.3
SAME_AREA_KM = 15.0


def distance_km(lat1, lon1, lat2, lon2):
    # Equirectangular approximation — fine at station scale, cheap enough to
    # run every addition against every qualified stop without an index.
    mean_lat = math.radians((lat1 + lat2) / 2)
    dx = math.radians(lon2 - lon1) * math.cos(mean_lat)
    dy = math.radians(lat2 - lat1)
    return math.hypot(dx, dy) * 6371.0


step5 = []
with open(
    local_input("step5_JoinedNTStops.csv", "step5_JoinNTStopsWithOSM.ipynb"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        if row["osm_lat"] and row["osm_lon"]:
            step5.append(
                {
                    "ontd_id": row["ontd_id"],
                    "stop_id": row["osm_stop_id"],
                    "name": row["osm_stop_name"] or row["ontd_name"],
                    "lat": float(row["osm_lat"]),
                    "lon": float(row["osm_lon"]),
                }
            )
qualified_ids = {s["stop_id"] for s in step5}
qualified_ontd = {s["ontd_id"] for s in step5}

duplicates, area_review = [], []
for stop_id, (label, reason, _infra) in ADDITIONS.items():
    lat = float(osm[stop_id]["stop_lat"])
    lon = float(osm[stop_id]["stop_lon"])
    d, nearest = min(
        ((distance_km(lat, lon, s["lat"], s["lon"]), s) for s in step5),
        key=lambda pair: pair[0],
    )
    if stop_id in qualified_ids:
        duplicates.append((label, stop_id, "same OSM id in step 5", nearest, d))
    elif ontd_of_osm.get(stop_id) in qualified_ontd:
        duplicates.append(
            (label, stop_id, "same ONTD station, different OSM object", nearest, d)
        )
    elif d <= SAME_STATION_KM:
        duplicates.append(
            (label, stop_id, f"{d * 1000:.0f} m from a qualified stop", nearest, d)
        )
    elif d <= SAME_AREA_KM and reason.startswith("fua"):
        area_review.append((label, stop_id, reason, nearest, d))

if duplicates:
    for label, stop_id, why, nearest, d in sorted(duplicates, key=lambda x: x[0]):
        print(f"  {label[:36]:38} {why:44} vs {nearest['name'][:32]} {stop_id}")
    raise ValueError(
        f"{len(duplicates)} addition(s) duplicate a step 5 stop — remove them, "
        "or replace the reason and id if a genuinely distinct station is meant"
    )

with open(OVERLAP_REVIEW_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "stop_id",
            "stop_name",
            "reason",
            "nearest_qualified",
            "nearest_stop_id",
            "distance_km",
        ],
    )
    writer.writeheader()
    for label, stop_id, reason, nearest, d in sorted(area_review, key=lambda x: -x[4]):
        writer.writerow(
            {
                "stop_id": stop_id,
                "stop_name": label,
                "reason": reason,
                "nearest_qualified": nearest["name"],
                "nearest_stop_id": nearest["stop_id"],
                "distance_km": f"{d:.1f}",
            }
        )
if area_review:
    print(
        f"{len(area_review)} fua additions have a qualified stop within "
        f"{SAME_AREA_KM:.0f} km — judgement calls, not errors; kept, and listed "
        f"in {OVERLAP_REVIEW_PATH.name}. If the second station is wanted, say "
        'so in the reason ("network:..."); an fua claim the data contradicts '
        "helps nobody."
    )

# --- resolve and write ------------------------------------------------------
rows = []
for stop_id, (label, reason, infra) in ADDITIONS.items():
    station = osm[stop_id]
    rows.append(
        {
            "stop_id": stop_id,
            "stop_name": station["stop_name"].strip() or label,
            "country": (
                ontd_country.get(stop_id)
                or station["country"].strip().upper()
                or MANUAL_COUNTRY.get(stop_id, "")
            ),
            "stop_lat": station["stop_lat"],
            "stop_lon": station["stop_lon"],
            "reason": reason.strip(),
            "infra_versions": infra,
        }
    )
rows.sort(key=lambda r: (r["country"], r["stop_name"]))

no_country = [r for r in rows if not r["country"]]
if no_country:
    raise ValueError(
        f"{len(no_country)} addition(s) have no country — step 10 cannot derive "
        f"a timezone and would drop them: "
        f"{[(r['stop_id'], r['stop_name']) for r in no_country]}. "
        "Add them to MANUAL_COUNTRY above."
    )

with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "stop_id",
            "stop_name",
            "country",
            "stop_lat",
            "stop_lon",
            "reason",
            "infra_versions",
        ],
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"wrote {len(rows)} rows to {OUTPUT_PATH.name}")
print("by country:", Counter(r["country"] for r in rows).most_common(8))

## Unfilled reasons

Everything listed here is a stop in the public catalog that cannot yet explain
why it is there — empty reasons and never-replaced `<...>` template
placeholders alike.


In [14]:
QUALIFIED_REASONS = {"fua", "tourism", "ferry", "border", "network", "night_train_stop"}
DETAILED_REASONS = {"fua", "tourism", "ferry"}


def reason_problem(reason: str) -> str | None:
    if not reason:
        return "empty"
    prefix, _, detail = reason.partition(":")
    prefix = prefix.split(" ")[0].split("\u2014")[0].strip()
    if prefix not in QUALIFIED_REASONS:
        return f"unknown criterion {prefix!r}"
    if prefix in DETAILED_REASONS:
        detail = detail.strip()
        if not detail:
            return f"'{prefix}' names no place"
        if "<" in detail or ">" in detail:
            return "template placeholder never filled in"
    return None


problems = [
    (row, problem) for row in rows if (problem := reason_problem(row["reason"]))
]
print(f"{len(problems)} of {len(rows)} additions cannot explain themselves\n")
for row, problem in problems:
    print(
        f"  {row['country']}  {row['stop_name'][:40]:42} {row['stop_id']:22} {problem}"
    )

337 of 354 additions cannot explain themselves

  AL  Durrës                                     osm:n13895194677       template placeholder never filled in
  BA  Mostar                                     osm:n312280159         template placeholder never filled in
  BA  Sarajevo                                   osm:n942406094         template placeholder never filled in
  BA  Zenica                                     osm:n10212563520       template placeholder never filled in
  BE  Brugge                                     osm:n2929614444        template placeholder never filled in
  BE  Gent-Sint-Pieters                          osm:n1178257779        template placeholder never filled in
  BE  Kortrijk                                   osm:n21309047          template placeholder never filled in
  BE  La Louvière-Centre                         osm:n446059037         template placeholder never filled in
  BE  Mechelen-Nekkerspoel                       osm:n7261826908        template